# Chapter 4 & 5 통합 요약 - ARIMA & 고급 예측 기법

## 📚 빠른 개요

| Chapter | 주제 | 핵심 내용 | 난이도 |
|---------|------|-----------|--------|
| **Chapter 4** | ARIMA 모델 | 시계열 예측의 기본 모델 | ⭐⭐⭐⭐ |
| **Chapter 5** | 고급 예측 기법 | SARIMA, Prophet, LSTM | ⭐⭐⭐⭐⭐ |

---

## 📖 Chapter 4: ARIMA 모델

### 핵심 개념

**ARIMA(p, d, q) = AR + I + MA**

- **AR(p)**: AutoRegressive - 과거 값으로 예측
  - Y(t) = c + φ₁Y(t-1) + φ₂Y(t-2) + ... + φₚY(t-p) + ε(t)
  - PACF로 p 결정

- **I(d)**: Integrated - 차분 횟수
  - d=0: 정상 시계열
  - d=1: 1차 차분
  - d=2: 2차 차분

- **MA(q)**: Moving Average - 과거 오차로 예측
  - Y(t) = μ + ε(t) + θ₁ε(t-1) + θ₂ε(t-2) + ... + θqε(t-q)
  - ACF로 q 결정

### 빠른 구현

```python
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 1단계: ACF/PACF로 파라미터 결정
plot_acf(df['value'], lags=40)   # q 결정
plot_pacf(df['value'], lags=40)  # p 결정

# 2단계: 모델 학습
model = ARIMA(df['value'], order=(p, d, q))
fitted_model = model.fit()

# 3단계: 예측
forecast = fitted_model.forecast(steps=30)

# 4단계: 평가
print(fitted_model.summary())
print(f"AIC: {fitted_model.aic}")
print(f"BIC: {fitted_model.bic}")
```

### 파라미터 선택 가이드

| ACF 패턴 | PACF 패턴 | 모델 |
|----------|-----------|------|
| 지수 감소 | lag p에서 절단 | AR(p) |
| lag q에서 절단 | 지수 감소 | MA(q) |
| 지수 감소 | 지수 감소 | ARMA(p,q) |

### Auto ARIMA (자동 파라미터 선택)

```python
from pmdarima import auto_arima

# 최적 파라미터 자동 탐색
auto_model = auto_arima(
    df['value'],
    start_p=0, start_q=0,
    max_p=5, max_q=5,
    d=None,  # 자동 결정
    seasonal=False,
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore'
)

print(auto_model.summary())
print(f"최적 파라미터: {auto_model.order}")
```

### 모델 진단

```python
# 잔차 분석
residuals = fitted_model.resid

# 1. 잔차 플롯
plt.figure(figsize=(12, 4))
plt.plot(residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('잔차 플롯')
plt.show()

# 2. 잔차 ACF (백색잡음 확인)
plot_acf(residuals, lags=40)

# 3. Ljung-Box 검정 (잔차 독립성)
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(residuals, lags=10)
print(lb_test)
```

---

## 📖 Chapter 5: 고급 예측 기법

### 1. SARIMA - 계절성 ARIMA

**SARIMA(p,d,q)(P,D,Q)s**
- (p,d,q): 비계절 파라미터
- (P,D,Q): 계절 파라미터
- s: 계절 주기 (월별=12, 분기별=4)

```python
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 월별 데이터 (계절 주기 = 12)
model = SARIMAX(
    df['value'],
    order=(1, 1, 1),           # 비계절 (p,d,q)
    seasonal_order=(1, 1, 1, 12)  # 계절 (P,D,Q,s)
)
fitted = model.fit()

# 예측
forecast = fitted.forecast(steps=12)
```

**언제 사용?**
- ✅ 명확한 계절성 패턴 (월별 매출, 분기별 수요)
- ✅ 주기적으로 반복되는 패턴
- ❌ 불규칙한 패턴, 트렌드 변화가 심한 경우

---

### 2. Prophet - Facebook의 시계열 예측

**특징:**
- 추세 + 계절성 + 휴일 효과 자동 처리
- 결측치에 강건함
- 직관적인 파라미터 튜닝

```python
from prophet import Prophet

# 데이터 준비 (ds, y 컬럼 필수)
df_prophet = df.reset_index()
df_prophet.columns = ['ds', 'y']

# 모델 학습
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05  # 추세 변화 민감도
)
model.fit(df_prophet)

# 미래 데이터프레임 생성
future = model.make_future_dataframe(periods=30)

# 예측
forecast = model.predict(future)

# 시각화
model.plot(forecast)
model.plot_components(forecast)  # 추세, 계절성 분해
```

**장점:**
- 🚀 빠른 학습 속도
- 🎯 자동 계절성 탐지
- 📅 휴일 효과 반영 가능
- 📊 직관적인 시각화

**단점:**
- ⚠️ 단기 예측에는 부적합
- ⚠️ 복잡한 패턴 포착 한계

---

### 3. LSTM - 딥러닝 기반 예측

**특징:**
- 장기 의존성(Long-term dependency) 학습
- 비선형 패턴 포착
- 대용량 데이터에 강력

```python
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

# 1. 데이터 정규화
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['value']])

# 2. 시퀀스 데이터 생성
def create_sequences(data, seq_length=60):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, seq_length=60)

# 3. 모델 구축
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(60, 1)),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# 4. 학습
model.fit(X, y, batch_size=32, epochs=50, validation_split=0.2)

# 5. 예측
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)
```

**장점:**
- 🧠 복잡한 비선형 패턴 학습
- 📈 다변량 시계열 처리 가능
- 🎯 장기 의존성 포착

**단점:**
- ⚠️ 대량의 데이터 필요
- ⚠️ 학습 시간 오래 걸림
- ⚠️ 하이퍼파라미터 튜닝 복잡
- ⚠️ 해석 가능성 낮음

---

## 🔍 모델 비교 및 선택 가이드

### 모델별 특성 비교

| 모델 | 장점 | 단점 | 적합한 경우 |
|------|------|------|-------------|
| **ARIMA** | • 통계적 해석 가능<br>• 적은 데이터로 학습<br>• 빠른 학습 | • 선형 관계만 포착<br>• 계절성 처리 어려움 | • 단순한 추세<br>• 비계절 데이터<br>• 통계적 근거 필요 |
| **SARIMA** | • 계절성 자동 처리<br>• ARIMA의 확장<br>• 해석 가능 | • 파라미터 많음<br>• 계산 복잡 | • 명확한 계절성<br>• 월별/분기별 데이터 |
| **Prophet** | • 사용 간편<br>• 휴일 효과<br>• 결측치 강건 | • 단기 예측 부적합<br>• 커스터마이징 한계 | • 일별 데이터<br>• 휴일 효과 중요<br>• 빠른 프로토타입 |
| **LSTM** | • 비선형 패턴<br>• 다변량 처리<br>• 장기 의존성 | • 대량 데이터 필요<br>• 학습 시간 김<br>• 해석 어려움 | • 복잡한 패턴<br>• 대용량 데이터<br>• 다변량 시계열 |

### 의사결정 트리

```
데이터 크기는?
├─ 작음 (< 100개)
│  └─ ARIMA / SARIMA
│
└─ 큼 (> 1000개)
   ├─ 계절성 있음?
   │  ├─ Yes → SARIMA / Prophet
   │  └─ No → ARIMA / LSTM
   │
   ├─ 비선형 패턴?
   │  ├─ Yes → LSTM
   │  └─ No → ARIMA / Prophet
   │
   └─ 해석 가능성 중요?
      ├─ Yes → ARIMA / SARIMA
      └─ No → LSTM
```

---

## 📊 모델 평가 지표

### 주요 평가 지표

```python
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# 1. RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"RMSE: {rmse:.2f}")

# 2. MAE (Mean Absolute Error)
mae = mean_absolute_error(y_true, y_pred)
print(f"MAE: {mae:.2f}")

# 3. MAPE (Mean Absolute Percentage Error)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
print(f"MAPE: {mape:.2f}%")

# 4. AIC / BIC (모델 복잡도 고려)
print(f"AIC: {model.aic:.2f}")
print(f"BIC: {model.bic:.2f}")
```

### 평가 지표 선택 가이드

| 지표 | 특징 | 언제 사용? |
|------|------|------------|
| **RMSE** | 큰 오차에 민감 | 이상치 중요할 때 |
| **MAE** | 모든 오차 동등 | 평균적 성능 평가 |
| **MAPE** | 상대적 오차 (%) | 스케일 다른 데이터 비교 |
| **AIC/BIC** | 모델 복잡도 페널티 | 모델 선택 시 |

---

## 🎯 실전 예제: 모델 앙상블

### 여러 모델을 결합하여 예측 성능 향상

```python
# 1. 각 모델로 예측
arima_pred = arima_model.forecast(steps=30)
prophet_pred = prophet_model.predict(future)['yhat'][-30:]
lstm_pred = lstm_model.predict(X_test)

# 2. 가중 평균 앙상블
ensemble_pred = (
    0.4 * arima_pred + 
    0.3 * prophet_pred + 
    0.3 * lstm_pred
)

# 3. 성능 비교
print(f"ARIMA RMSE: {rmse(y_true, arima_pred):.2f}")
print(f"Prophet RMSE: {rmse(y_true, prophet_pred):.2f}")
print(f"LSTM RMSE: {rmse(y_true, lstm_pred):.2f}")
print(f"Ensemble RMSE: {rmse(y_true, ensemble_pred):.2f}")
```

**앙상블 전략:**
- **평균 앙상블**: 모든 모델 동등 가중치
- **가중 평균**: 성능 좋은 모델에 높은 가중치
- **스태킹**: 메타 모델로 최종 예측

---

##  핵심 요약

### Chapter 4 (ARIMA) 핵심
1. **ACF/PACF로 파라미터 결정**
2. **AIC/BIC로 모델 선택**
3. **잔차 분석으로 모델 진단**
4. **Auto ARIMA로 자동화 가능**

### Chapter 5 (고급 기법) 핵심
1. **SARIMA**: 계절성 있는 데이터
2. **Prophet**: 빠른 프로토타입, 휴일 효과
3. **LSTM**: 복잡한 비선형 패턴
4. **앙상블**: 여러 모델 결합으로 성능 향상

### 실무 체크리스트
- [ ] 데이터 특성 파악 (크기, 계절성, 패턴)
- [ ] 적절한 모델 선택
- [ ] 정상성 확보 (차분)
- [ ] 파라미터 튜닝
- [ ] 교차 검증
- [ ] 잔차 분석
- [ ] 여러 지표로 평가
- [ ] 앙상블 고려

---

학습 시간: Chapter 4 (6-8시간) + Chapter 5 (8-10시간)  
